In [70]:
# Import everything needed for ReAct agents
import os  # Read API keys from environment variables

from langchain_classic import hub  # Pull the standard ReAct prompt template
from langchain_groq import ChatGroq  # Groq chat model wrapper
from langchain_classic.agents import Tool, AgentExecutor, create_react_agent  # ReAct building blocks
from langchain_experimental.utilities import PythonREPL  # Safe Python runner for agents
from langchain_community.utilities import GoogleSerperAPIWrapper  # Serper search wrapper


In [71]:
serper_api_key=os.getenv("SERPER_API_KEY")
groq_api_key=os.getenv("api_key")

# adding the brain of the agent ( the llm )
groq_llm=ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0
)


In [ ]:
# Creating a python repl to run python code in python environment
python_repl=PythonREPL()
python_repl.run(
    "print('Hi there how are you doing')"
)

repl_tool=Tool(
    name="python_repl",

    description=(
        "a pyhton shell to execute the python commands"
        "Input should be a valid python queries"
    ),
    func=python_repl.run,# Actual function that runs wwhen agents invoke the tool
)
# Testing the tool by stimulating the invokations
repl_tool.invoke(
    "print(3+4)"
)

'7\n'

In [73]:
# Creating a searcher for searching on google for us
serper_search=GoogleSerperAPIWrapper(serper_api_key=serper_api_key)
# testing the tool searcher
print(serper_search.run("what is name of capital of india?"))

# creating the tool
serper_tool=Tool(
    name="serper_search",
    description=(
        "An Interface to serper searrch Engine , and input should be a string"
    ),
    func=serper_search.run
)
# stimulating the invoke of the tool as the agent would
serper_tool.invoke("who is elone musk?")


New Delhi


"Elon Musk: Businessman and former Senior Advisor to the President of the United States. Elon Musk Born: June 28, 1971 (age 54 years), Pretoria, South Africa. Elon Musk Children: Vivian Jenna Wilson, Nevada Alexander Musk, Griffin Musk, and more. Elon Musk Spouse: Talulah Riley (m. 2013–2016), Talulah Riley (m. 2010–2012), and Justine Musk (m. 2000–2008). Elon Musk Education: University of Pennsylvania School of Arts and Sciences (1997), Wharton School (1997), Queen's University (1989–1991), and more. Elon Musk Parents: Errol Musk and Maye Musk. Elon Reeve Musk is a businessman and former public official known for his leadership of Tesla and SpaceX. Musk has been the wealthiest person in the world ... Elon Reeve Musk is a US-based celebrity, business magnate, and investor. He was born on June 28, 1971 (Elon Musk's age is 52 years old as of his birthday in ... The boss of X, Tesla and SpaceX, already the world's richest person, is now also its first trillionaire. As the co-founder and C

In [76]:
# Pulling the system prompt for the reAct agent
# Manually define the ReAct template
from langchain_classic.prompts import PromptTemplate
template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Question: {input}
Thought: {agent_scratchpad}"""

react_prompt = PromptTemplate.from_template(template)

react_agent = create_react_agent(
    llm=groq_llm,
    tools=[repl_tool],
    prompt=react_prompt
)

agent_executer=AgentExecutor(
    agent=react_agent,
    tools=[repl_tool],
    verbose=True
)

In [78]:
user_input = (
    "If $ 450 amounts to $ 630 in 6 years, what will it amount to in 2 years "
    "at the same interest rate?"
)

response=agent_executer.invoke(
    {"input":user_input}
)
print(response["output"])



> Entering new AgentExecutor chain...
Thought: To solve this problem, we first need to find the interest rate at which $450 amounts to $630 in 6 years. We can use the formula for compound interest: A = P(1 + r)^n, where A is the amount after n years, P is the principal amount, r is the annual interest rate, and n is the number of years.

Action: python_repl
Action Input: ```python
import math
# Given values
P = 450  # Principal amount
A = 630  # Amount after 6 years
n = 6    # Number of years

# Calculate the interest rate
r = (A / P) ** (1 / n) - 1
print(r)
```0.05768092640521627
Now that we have the interest rate, we can use it to find the amount after 2 years. We will use the same formula for compound interest: A = P(1 + r)^n, where A is the amount after n years, P is the principal amount, r is the annual interest rate, and n is the number of years.

Action: python_repl
Action Input: ```python
import math
# Given values
P = 450  # Principal amount
r = 0.05768092640521627  # Intere